In [70]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss

In [71]:
TRAIN_CUTOFF = 2003

teams   = pd.read_csv('data/mania_26/MTeams.csv')
seeds   = pd.read_csv('data/mania_26/MNCAATourneySeeds.csv')
results = pd.read_csv('data/mania_26/MNCAATourneyCompactResults.csv')
reg_det = pd.read_csv('data/mania_26/MRegularSeasonDetailedResults.csv')
reg     = pd.read_csv('data/mania_26/MRegularSeasonCompactResults.csv')

In [72]:
TOURNEY_TO_KENPOM = {
    'Abilene Chr': 'Abilene Christian', 'SUNY Albany': 'Albany',
    'American Univ': 'American', 'Ark Little Rock': 'Arkansas Little Rock',
    'Ark Pine Bluff': 'Arkansas Pine Bluff', 'Bethune-Cookman': 'Bethune Cookman',
    'Birmingham So': 'Birmingham Southern', 'Boston Univ': 'Boston University',
    'Brooklyn': 'LIU Brooklyn', 'C Michigan': 'Central Michigan',
    'Cent Arkansas': 'Central Arkansas', 'Central Conn': 'Central Connecticut',
    'Charleston So': 'Charleston Southern', 'Citadel': 'The Citadel',
    'Coastal Car': 'Coastal Carolina', 'Col Charleston': 'Charleston',
    'College of Charleston': 'Charleston', 'CS Bakersfield': 'Cal St. Bakersfield',
    'CS Fullerton': 'Cal St. Fullerton', 'CS Northridge': 'Cal St. Northridge',
    'CS Sacramento': 'Sacramento St.', 'E Illinois': 'Eastern Illinois',
    'E Kentucky': 'Eastern Kentucky', 'E Michigan': 'Eastern Michigan',
    'E Washington': 'Eastern Washington', 'ETSU': 'East Tennessee St.',
    'F Dickinson': 'Fairleigh Dickinson', 'FL Atlantic': 'Florida Atlantic',
    'FGCU': 'Florida Gulf Coast', 'Florida Intl': 'FIU',
    'G Washington': 'George Washington', 'Ga Southern': 'Georgia Southern',
    'Grambling': 'Grambling St.', 'Houston Chr': 'Houston Christian',
    'IL Chicago': 'Illinois Chicago', 'PFW': 'Purdue Fort Wayne',
    'Kennesaw': 'Kennesaw St.', 'Kent': 'Kent St.',
    'Loy Marymount': 'Loyola Marymount', 'Loyola-Chicago': 'Loyola Chicago',
    'MA Lowell': 'UMass Lowell', 'MD E Shore': 'Maryland Eastern Shore',
    'Missouri KC': 'UMKC', 'Monmouth NJ': 'Monmouth',
    'MS Valley St': 'Mississippi Valley St.', 'MS Valley St.': 'Mississippi Valley St.',
    'Mt St Mary': "Mount St. Mary's", "Mt St Mary's": "Mount St. Mary's",
    'MTSU': 'Middle Tennessee', 'N Colorado': 'Northern Colorado',
    'N Dakota St': 'North Dakota St.', 'N Dakota St.': 'North Dakota St.',
    'N Illinois': 'Northern Illinois', 'N Kentucky': 'Northern Kentucky',
    'NC A&T': 'North Carolina A&T', 'NC Central': 'North Carolina Central',
    'NE Omaha': 'Nebraska Omaha', 'Northwestern LA': 'Northwestern St.',
    'Prairie View': 'Prairie View A&M', 'S Carolina St': 'South Carolina St.',
    'S Carolina St.': 'South Carolina St.', 'S Dakota St': 'South Dakota St.',
    'S Dakota St.': 'South Dakota St.', 'S Illinois': 'Southern Illinois',
    'SC Upstate': 'USC Upstate', 'SE Louisiana': 'Southeastern Louisiana',
    'SE Missouri St': 'Southeast Missouri St.', 'SE Missouri St.': 'Southeast Missouri St.',
    'SF Austin': 'Stephen F. Austin', 'Southern Univ': 'Southern',
    'St Bonaventure': 'St. Bonaventure', 'St Francis NY': 'St. Francis NY',
    'St Francis PA': 'St. Francis PA', "St John's": "St. John's",
    "St Joseph's PA": "Saint Joseph's", 'St Louis': 'Saint Louis',
    "St Mary's CA": "Saint Mary's", "St Peter's": "Saint Peter's",
    'TAM C. Christi': 'Texas A&M Corpus Chris', 'TN Martin': 'Tennessee Martin',
    'UTRGV': 'UT Rio Grande Valley', 'TX Southern': 'Texas Southern',
    'ULM': 'Louisiana Monroe', 'UT San Antonio': 'UTSA',
    'W Carolina': 'Western Carolina', 'W Illinois': 'Western Illinois',
    'WKU': 'Western Kentucky', 'W Michigan': 'Western Michigan',
    'W Salem St.': 'Winston Salem St.', 'WI Green Bay': 'Green Bay',
    'WI Milwaukee': 'Milwaukee', 'St Thomas MN': 'St. Thomas', 'Queens NC': 'Queens',
}

teams['KenPomName'] = teams['TeamName'].str.replace(r' St$', ' St.', regex=True)
teams['KenPomName'] = teams['KenPomName'].map(lambda x: TOURNEY_TO_KENPOM.get(x, x))
print('Teams mapped:', len(teams))

Teams mapped: 381


In [73]:
def compute_team_stats(reg_df):
    # Winner perspective
    w = reg_df.copy()
    w['TeamID'] = w['WTeamID']; w['Pts'] = w['WScore']; w['OppPts'] = w['LScore']
    for c in ['FGM','FGA','FGM3','FGA3','FTM','FTA','OR','DR','Ast','TO','Stl','Blk']:
        w[c] = w[f'W{c}']
    w['OppFGM']=w['LFGM']; w['OppFGA']=w['LFGA']; w['OppFGM3']=w['LFGM3']
    w['OppFGA3']=w['LFGA3']; w['OppFTA']=w['LFTA']; w['OppOR']=w['LOR']
    w['OppAst']=w['LAst'];  w['OppTO']=w['LTO']

    # Loser perspective
    l = reg_df.copy()
    l['TeamID'] = l['LTeamID']; l['Pts'] = l['LScore']; l['OppPts'] = l['WScore']
    for c in ['FGM','FGA','FGM3','FGA3','FTM','FTA','OR','DR','Ast','TO','Stl','Blk']:
        l[c] = l[f'L{c}']
    l['OppFGM']=l['WFGM']; l['OppFGA']=l['WFGA']; l['OppFGM3']=l['WFGM3']
    l['OppFGA3']=l['WFGA3']; l['OppFTA']=l['WFTA']; l['OppOR']=l['WOR']
    l['OppAst']=l['WAst'];  l['OppTO']=l['WTO']

    cols = ['Season','TeamID','Pts','OppPts','FGM','FGA','FGM3','FGA3','FTM','FTA',
            'OR','Ast','TO','Stl','Blk','OppFGM','OppFGA','OppFGM3','OppFGA3',
            'OppFTA','OppOR','OppAst','OppTO']
    games = pd.concat([w[cols], l[cols]], ignore_index=True)

    games['Poss']    = games['FGA']    - games['OR']    + games['TO']    + 0.475*games['FTA']
    games['OppPoss'] = games['OppFGA'] - games['OppOR'] + games['OppTO'] + 0.475*games['OppFTA']

    agg_cols = ['Pts','OppPts','Poss','OppPoss','FGM','FGA','FGM3','FGA3','FTM','FTA',
                'OR','Ast','TO','Stl','Blk','OppFGM','OppFGA','OppFGM3','OppFGA3','OppOR','OppAst']
    agg = games.groupby(['Season','TeamID'])[agg_cols].sum().reset_index()
    gc  = games.groupby(['Season','TeamID']).size().reset_index(name='G')
    agg = agg.merge(gc, on=['Season','TeamID'])

    # Efficiency
    agg['OE']    = 100 * agg['Pts']    / agg['Poss']
    agg['DE']    = 100 * agg['OppPts'] / agg['OppPoss']
    agg['AdjEM'] = agg['OE'] - agg['DE']
    agg['AdjOE'] = agg['OE']
    agg['AdjDE'] = agg['DE']
    agg['Tempo'] = agg['Poss'] / agg['G']

    # Shooting
    agg['FG2Pct']  = (agg['FGM'] - agg['FGM3']) / (agg['FGA'] - agg['FGA3'])
    agg['FG3Pct']  = agg['FGM3']  / agg['FGA3']
    agg['FTPct']   = agg['FTM']   / agg['FTA']
    agg['FG3Rate'] = agg['FGA3']  / agg['FGA']
    agg['FTRate']  = agg['FTA']   / agg['FGA']

    # Offensive
    agg['ARate']    = agg['Ast']  / agg['FGM']
    agg['StlRate']  = agg['Stl']  / agg['OppPoss']
    agg['BlockPct'] = agg['Blk']  / (agg['OppFGA'] - agg['OppFGA3'])
    agg['ORPct']    = agg['OR']   / (agg['OR'] + agg['OppOR'])
    agg['TOPct']    = agg['TO']   / agg['Poss']

    # Defensive
    agg['OppFG2Pct']  = (agg['OppFGM'] - agg['OppFGM3']) / (agg['OppFGA'] - agg['OppFGA3'])
    agg['OppFG3Pct']  = agg['OppFGM3'] / agg['OppFGA3']
    agg['OppFG3Rate'] = agg['OppFGA3'] / agg['OppFGA']
    agg['OppARate']   = agg['OppAst']  / agg['OppFGM']

    agg.drop(columns=['Pts','OppPts','Poss','OppPoss'], inplace=True)

    # ── Strength of Schedule ──
    team_em = agg[['Season','TeamID','AdjEM']].copy()
    opp_w = (reg_df
        .merge(team_em, left_on=['Season','LTeamID'], right_on=['Season','TeamID'])
        [['Season','WTeamID','AdjEM']]
    )
    opp_l = (reg_df
        .merge(team_em, left_on=['Season','WTeamID'], right_on=['Season','TeamID'])
        [['Season','LTeamID','AdjEM']]
    )
    sos_w = opp_w.groupby(['Season','WTeamID'])['AdjEM'].mean().reset_index().rename(columns={'WTeamID':'TeamID','AdjEM':'SOS'})
    sos_l = opp_l.groupby(['Season','LTeamID'])['AdjEM'].mean().reset_index().rename(columns={'LTeamID':'TeamID','AdjEM':'SOS'})
    sos   = pd.concat([sos_w, sos_l]).groupby(['Season','TeamID'])['SOS'].mean().reset_index()
    agg   = agg.merge(sos, on=['Season','TeamID'], how='left')

    return agg

box_stats = compute_team_stats(reg_det)
box_stats = box_stats.merge(teams[['TeamID','KenPomName']], on='TeamID', how='left')
box_stats = box_stats.rename(columns={'KenPomName': 'TeamName'})

In [74]:
def fit_bradley_terry(season_games, max_iter=300, tol=1e-8):
    teams_list = sorted(set(season_games['WTeamID']) | set(season_games['LTeamID']))
    n   = len(teams_list)
    idx = {t: i for i, t in enumerate(teams_list)}

    wins_arr = np.array([idx[t] for t in season_games['WTeamID']])
    loss_arr = np.array([idx[t] for t in season_games['LTeamID']])

    win_counts = np.zeros(n)
    np.add.at(win_counts, wins_arr, 1)

    s = np.ones(n)
    for _ in range(max_iter):
        s_old    = s.copy()
        denom    = np.zeros(n)
        pair_sum = s[wins_arr] + s[loss_arr]
        np.add.at(denom, wins_arr, 1.0 / pair_sum)
        np.add.at(denom, loss_arr, 1.0 / pair_sum)
        s = win_counts / np.where(denom > 0, denom, 1e-10)
        s /= np.exp(np.mean(np.log(s + 1e-10)))
        if np.max(np.abs(s - s_old)) < tol:
            break

    log_s = np.log(s + 1e-10)
    return {teams_list[i]: log_s[i] for i in range(n)}

def compute_bt_ratings(reg_df):
    all_ratings = []
    for season, grp in reg_df.groupby('Season'):
        ratings = fit_bradley_terry(grp)
        for tid, rating in ratings.items():
            all_ratings.append({'Season': season, 'TeamID': tid, 'BT': rating})
    return pd.DataFrame(all_ratings)

bt_ratings = compute_bt_ratings(reg)
bt_ratings = (bt_ratings
    .merge(teams[['TeamID','KenPomName']], on='TeamID', how='left')
    .rename(columns={'KenPomName': 'TeamName'}))

In [75]:
wins   = reg.groupby(['Season','WTeamID']).size().reset_index(name='Wins')
losses = reg.groupby(['Season','LTeamID']).size().reset_index(name='Losses')
record = wins.merge(losses, left_on=['Season','WTeamID'], right_on=['Season','LTeamID'], how='outer')
record['Record'] = record['Wins'].fillna(0) / (record['Wins'].fillna(0) + record['Losses'].fillna(0))
record = record.rename(columns={'WTeamID':'TeamID'})[['Season','TeamID','Record']]

In [76]:
seeds['SeedNum'] = seeds['Seed'].str.extract(r'(\d+)').astype(int)

games = (
    results[results['Season'] >= 2003].copy()
    .merge(seeds.rename(columns={'TeamID':'WTeamID','SeedNum':'WSeed'})[['Season','WTeamID','WSeed']], on=['Season','WTeamID'])
    .merge(seeds.rename(columns={'TeamID':'LTeamID','SeedNum':'LSeed'})[['Season','LTeamID','LSeed']], on=['Season','LTeamID'])
    .merge(teams[['TeamID','KenPomName']].rename(columns={'TeamID':'WTeamID','KenPomName':'WTeamName'}), on='WTeamID')
    .merge(teams[['TeamID','KenPomName']].rename(columns={'TeamID':'LTeamID','KenPomName':'LTeamName'}), on='LTeamID')
    .merge(record.rename(columns={'TeamID':'WTeamID','Record':'WRecord'}), on=['Season','WTeamID'], how='left')
    .merge(record.rename(columns={'TeamID':'LTeamID','Record':'LRecord'}), on=['Season','LTeamID'], how='left')
)

def higher_seed(row):
    if row['WSeed'] < row['LSeed']:      return row['WTeamName']
    if row['LSeed'] < row['WSeed']:      return row['LTeamName']
    if row['WRecord'] >= row['LRecord']: return row['WTeamName']
    return row['LTeamName']

games['HigherSeed']    = games.apply(higher_seed, axis=1)
games['HigherSeedWin'] = (games['HigherSeed'] == games['WTeamName']).astype(int)
games['HigherSeedNum'] = games[['WSeed','LSeed']].min(axis=1)
games['LowerSeedNum']  = games[['WSeed','LSeed']].max(axis=1)
games['HigherRecord']  = games.apply(lambda r: r['WRecord'] if r['HigherSeed']==r['WTeamName'] else r['LRecord'], axis=1)
games['LowerRecord']   = games.apply(lambda r: r['LRecord'] if r['HigherSeed']==r['WTeamName'] else r['WRecord'], axis=1)

GAMES_KEEP = ['Season','WTeamName','LTeamName','HigherSeed','HigherSeedWin',
              'HigherSeedNum','LowerSeedNum','HigherRecord','LowerRecord']
games_clean = games[GAMES_KEEP].copy()

In [77]:
matchup = (
    games_clean
    .merge(box_stats, left_on=['Season','WTeamName'], right_on=['Season','TeamName'], how='left')
    .merge(box_stats, left_on=['Season','LTeamName'], right_on=['Season','TeamName'], suffixes=('_W','_L'), how='left')
)
matchup.drop(columns=['TeamName_W','TeamName_L','TeamID_W','TeamID_L','G_W','G_L'], inplace=True, errors='ignore')

na_pct = matchup.isna().mean()
drop_cols = na_pct[na_pct > 0.01].index.tolist()
matchup.drop(columns=drop_cols, inplace=True)

cols_w = [c for c in matchup.columns if c.endswith('_W')]
cols_l = [c for c in matchup.columns if c.endswith('_L')]
matchup[cols_w + cols_l] = matchup[cols_w + cols_l].apply(pd.to_numeric, errors='coerce')

for col_w in cols_w:
    col_l = col_w.replace('_W','_L')
    if col_l in cols_l:
        stat = col_w.replace('_W','')
        higher_wins = matchup['HigherSeed'] == matchup['WTeamName']
        matchup[f'{stat}_diff'] = np.where(
            higher_wins,
            matchup[col_w] - matchup[col_l],
            matchup[col_l] - matchup[col_w]
        )

matchup.drop(columns=cols_w + cols_l, inplace=True)

bt_w = bt_ratings[['Season','TeamName','BT']].rename(columns={'TeamName':'WTeamName','BT':'BT_W'})
bt_l = bt_ratings[['Season','TeamName','BT']].rename(columns={'TeamName':'LTeamName','BT':'BT_L'})
matchup = matchup.merge(bt_w, on=['Season','WTeamName'], how='left')
matchup = matchup.merge(bt_l, on=['Season','LTeamName'], how='left')
higher_wins = matchup['HigherSeed'] == matchup['WTeamName']
matchup['BT_diff'] = np.where(higher_wins,
    matchup['BT_W'] - matchup['BT_L'],
    matchup['BT_L'] - matchup['BT_W']
)
matchup.drop(columns=['BT_W','BT_L'], inplace=True)

In [78]:
diff_cols_check = [c for c in matchup.columns if c.endswith('_diff')]
features_check  = ['HigherSeedNum','LowerSeedNum','HigherRecord','LowerRecord'] + diff_cols_check
val_years_check = [2022, 2023, 2024, 2025]
cutoffs_to_test = [2003, 2005, 2007, 2009, 2011, 2013, 2015, 2017]

cutoff_scores = {}
for cutoff in cutoffs_to_test:
    scores = []
    for val_yr in val_years_check:
        train = matchup[(matchup['Season'] >= cutoff) & (matchup['Season'] < val_yr)]
        test  = matchup[matchup['Season'] == val_yr]
        rf_   = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1)
        rf_.fit(train[features_check].fillna(0), train['HigherSeedWin'])
        probs = rf_.predict_proba(test[features_check].fillna(0))[:, 1]
        scores.append(brier_score_loss(test['HigherSeedWin'], probs))
    avg = np.mean(scores)
    cutoff_scores[cutoff] = avg
    n_s = len(matchup[(matchup['Season'] >= cutoff) & (matchup['Season'] < 2022)]['Season'].unique())
    best_so_far = cutoff == min(cutoff_scores, key=cutoff_scores.get)
    print(f"{cutoff:<8} {n_s:>10} {scores[0]:>8.4f} {scores[1]:>8.4f} {scores[2]:>8.4f} {scores[3]:>8.4f} {avg:>8.4f}")

best = min(cutoff_scores, key=cutoff_scores.get)

2003             18   0.2221   0.2066   0.1849   0.1451   0.1897
2005             16   0.2199   0.2068   0.1891   0.1447   0.1901
2007             14   0.2241   0.2074   0.1843   0.1463   0.1905
2009             12   0.2172   0.2095   0.1858   0.1498   0.1906
2011             10   0.2190   0.2086   0.1887   0.1500   0.1916
2013              8   0.2226   0.2103   0.1876   0.1516   0.1930
2015              6   0.2255   0.2072   0.1906   0.1524   0.1939
2017              4   0.2234   0.2103   0.1885   0.1516   0.1935


In [79]:
importance_acc = {}
for val_yr in [2022, 2023, 2024, 2025]:
    train = matchup[(matchup['Season'] >= TRAIN_CUTOFF) & (matchup['Season'] < val_yr)]
    rf_   = RandomForestClassifier(n_estimators=400, max_depth=6, random_state=42, n_jobs=-1)
    rf_.fit(train[features_check].fillna(0), train['HigherSeedWin'])
    for feat, imp in zip(features_check, rf_.feature_importances_):
        importance_acc[feat] = importance_acc.get(feat, 0) + imp

avg_imp = pd.Series({k: v/4 for k, v in importance_acc.items()}).sort_values(ascending=False)
ranked_features = list(avg_imp.index)

cumulative = 0
for feat, imp in avg_imp.items():
    cumulative += imp
    print(f'  {feat:<22} {imp:>10.4f}')

print()

best_n = None; best_avg_trim = 999
for n_feat in [10, 12, 15, 18, 20, 25, 30, len(features_check)]:
    feats  = ranked_features[:n_feat]
    scores = []
    for val_yr in [2022, 2023, 2024, 2025]:
        train = matchup[(matchup['Season'] >= TRAIN_CUTOFF) & (matchup['Season'] < val_yr)]
        test  = matchup[matchup['Season'] == val_yr]
        rf_   = RandomForestClassifier(n_estimators=400, max_depth=6, random_state=42, n_jobs=-1)
        rf_.fit(train[feats].fillna(0), train['HigherSeedWin'])
        probs = rf_.predict_proba(test[feats].fillna(0))[:, 1]
        scores.append(brier_score_loss(test['HigherSeedWin'], probs))
    avg = np.mean(scores)
    if avg < best_avg_trim:
        best_avg_trim = avg; best_n = n_feat
    label  = f'Top {n_feat}' if n_feat < len(features_check) else f'All {n_feat}'
    print(f'{label:<12} {scores[0]:>8.4f} {scores[1]:>8.4f} {scores[2]:>8.4f} {scores[3]:>8.4f} {avg:>8.4f}')

FEATURES_TRIMMED = ranked_features[:best_n]

  BT_diff                    0.0935
  SOS_diff                   0.0659
  AdjEM_diff                 0.0536
  LowerSeedNum               0.0362
  AdjOE_diff                 0.0325
  OE_diff                    0.0318
  FGM_diff                   0.0288
  HigherRecord               0.0277
  Tempo_diff                 0.0247
  StlRate_diff               0.0238
  ARate_diff                 0.0232
  BlockPct_diff              0.0232
  Stl_diff                   0.0232
  FTRate_diff                0.0229
  OppFG3Pct_diff             0.0219
  Blk_diff                   0.0219
  DE_diff                    0.0211
  HigherSeedNum              0.0210
  Ast_diff                   0.0208
  FGA_diff                   0.0197
  ORPct_diff                 0.0194
  OppARate_diff              0.0194
  AdjDE_diff                 0.0188
  FTPct_diff                 0.0187
  LowerRecord                0.0181
  OR_diff                    0.0178
  FG2Pct_diff                0.0177
  OppAst_diff               

In [86]:
diff_cols = [c for c in matchup.columns if c.endswith('_diff')]

features_all = [
    'HigherSeedNum', 'LowerSeedNum', 'HigherRecord', 'LowerRecord',
] + diff_cols

features_trimmed = FEATURES_TRIMMED

features = features_trimmed

val_season    = 2025
test_season   = 2026
train_seasons = [s for s in matchup['Season'].unique()
                 if s >= TRAIN_CUTOFF and s not in (val_season, test_season)]

x_train = matchup[matchup['Season'].isin(train_seasons)][features].fillna(0)
y_train = matchup[matchup['Season'].isin(train_seasons)]['HigherSeedWin']
x_val   = matchup[matchup['Season'] == val_season][features].fillna(0)
y_val   = matchup[matchup['Season'] == val_season]['HigherSeedWin']

In [90]:
rf = RandomForestClassifier(n_estimators=400, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(x_train, y_train)

val_probs_rf = rf.predict_proba(x_val)[:, 1]
print(f'RF Brier Score (val {val_season}): {brier_score_loss(y_val, val_probs_rf):.4f}')
print()

importance = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(importance.head(15).round(4))

RF Brier Score (val 2025): 0.1474

BT_diff           0.1189
SOS_diff          0.0739
AdjEM_diff        0.0568
LowerSeedNum      0.0426
AdjOE_diff        0.0417
OE_diff           0.0385
FGM_diff          0.0321
HigherRecord      0.0315
FTRate_diff       0.0313
Tempo_diff        0.0309
ARate_diff        0.0300
BlockPct_diff     0.0296
OppFG3Pct_diff    0.0289
StlRate_diff      0.0281
Blk_diff          0.0277
dtype: float64


In [91]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    booster='gbtree', objective='binary:logistic',
    n_estimators=400, max_depth=7, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=4, gamma=1, reg_lambda=2, reg_alpha=1,
    random_state=42, n_jobs=-1, eval_metric='logloss',
)
xgb.fit(x_train, y_train)

val_probs_xgb = xgb.predict_proba(x_val)[:, 1]
print(f'XGB Brier Score (val {val_season}): {brier_score_loss(y_val, val_probs_xgb):.4f}')

XGB Brier Score (val 2025): 0.1664


In [92]:
VAL_SEASONS = [2022, 2023, 2024]
HOLDOUT     = 2025

def train_models(train_data):
    X = train_data[features].fillna(0)
    y = train_data['HigherSeedWin']
    rf_ = RandomForestClassifier(n_estimators=400, max_depth=6, random_state=42, n_jobs=-1)
    rf_.fit(X, y)
    xgb_ = XGBClassifier(
        booster='gbtree', objective='binary:logistic',
        n_estimators=400, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=4, gamma=1, reg_lambda=2, reg_alpha=1,
        random_state=42, n_jobs=-1, eval_metric='logloss',
    )
    xgb_.fit(X, y)
    return rf_, xgb_

season_data = {}
for s in VAL_SEASONS + [HOLDOUT]:
    train_data = matchup[(matchup['Season'] >= TRAIN_CUTOFF) & (matchup['Season'] < s)]
    rf_, xgb_  = train_models(train_data)
    test_data  = matchup[matchup['Season'] == s]
    X_test     = test_data[features].fillna(0)
    season_data[s] = {
        'rf':  rf_.predict_proba(X_test)[:, 1],
        'xgb': xgb_.predict_proba(X_test)[:, 1],
        'y':   test_data['HigherSeedWin'].values,
    }
    rf_b  = brier_score_loss(season_data[s]['y'], season_data[s]['rf'])
    xgb_b = brier_score_loss(season_data[s]['y'], season_data[s]['xgb'])

best_avg = 999; best_w = (1.0, 0.0)
all_res  = []
for w_rf in np.arange(0.0, 1.05, 0.05).round(2):
    w_xgb  = round(1.0 - w_rf, 2)
    scores = [brier_score_loss(season_data[s]['y'],
               w_rf*season_data[s]['rf'] + w_xgb*season_data[s]['xgb'])
              for s in VAL_SEASONS]
    avg = np.mean(scores)
    all_res.append((w_rf, w_xgb, avg, scores))
    if avg < best_avg:
        best_avg = avg; best_w = (w_rf, w_xgb)

for w_rf, w_xgb, avg, scores in all_res:
    print(f'  {w_rf:>6.2f} {w_xgb:>7.2f}  {scores[0]:>8.4f} {scores[1]:>8.4f} {scores[2]:>8.4f}  {avg:>8.4f}')

W_RF  = best_w[0]
W_XGB = best_w[1]
print(f'\nBest weights -> RF: {W_RF:.2f}, XGB: {W_XGB:.2f}')

W_RF  = 0.5
W_XGB = 0.5
print(f'Overriding weights -> RF: {W_RF:.2f}, XGB: {W_XGB:.2f}')

    0.00    1.00    0.2482   0.2137   0.1886    0.2168
    0.05    0.95    0.2459   0.2125   0.1881    0.2155
    0.10    0.90    0.2437   0.2113   0.1877    0.2142
    0.15    0.85    0.2416   0.2102   0.1874    0.2131
    0.20    0.80    0.2396   0.2092   0.1871    0.2120
    0.25    0.75    0.2377   0.2083   0.1868    0.2109
    0.30    0.70    0.2358   0.2074   0.1867    0.2100
    0.35    0.65    0.2340   0.2067   0.1865    0.2091
    0.40    0.60    0.2323   0.2059   0.1865    0.2083
    0.45    0.55    0.2307   0.2053   0.1865    0.2075
    0.50    0.50    0.2292   0.2048   0.1866    0.2068
    0.55    0.45    0.2277   0.2043   0.1867    0.2062
    0.60    0.40    0.2263   0.2039   0.1869    0.2057
    0.65    0.35    0.2250   0.2035   0.1871    0.2052
    0.70    0.30    0.2238   0.2033   0.1874    0.2048
    0.75    0.25    0.2226   0.2031   0.1878    0.2045
    0.80    0.20    0.2216   0.2030   0.1882    0.2042
    0.85    0.15    0.2206   0.2030   0.1887    0.2041
    0.90  

In [93]:
val_ensemble = W_RF * val_probs_rf + W_XGB * val_probs_xgb

print(f'Ensemble Brier  ({val_season}): {brier_score_loss(y_val, val_ensemble):.4f}')

Ensemble Brier  (2025): 0.1530


In [94]:
CSV_NAME_FIX = {
    'Iowa St':      'Iowa St.',
    'Kennesaw':     'Kennesaw St.',
    'McNeese St':   'McNeese St.',
    'Michigan St':  'Michigan St.',
    'N Dakota St':  'North Dakota St.',
    'Ohio St':      'Ohio St.',
    'Prairie View': 'Prairie View A&M',
    'Queens NC':    'Queens',
    "St John's":    "St. John's",
    'St Louis':     'Saint Louis',
    "St Mary's CA": "Saint Mary's",
    'Tennessee St': 'Tennessee St.',
    'Utah St':      'Utah St.',
    'Wright St':    'Wright St.',
}

def predict_bracket(matchup_csv_path):
    bracket = pd.read_csv(matchup_csv_path)
    print(f'Loaded {len(bracket)} matchups')

    # ── Normalise column names ──
    col_map = {
        'HigherSeed':    'higher_seed',
        'LowerSeed':     'lower_seed',
        'HigherSeedNum': 'higher_seed_num',
        'LowerSeedNum':  'lower_seed_num',
        'HigherSeedID':  'higher_seed_id',
        'LowerSeedID':   'lower_seed_id',
    }
    bracket = bracket.rename(columns=col_map)

    bracket['higher_seed'] = bracket['higher_seed'].map(lambda x: CSV_NAME_FIX.get(x, x))
    bracket['lower_seed']  = bracket['lower_seed'].map(lambda x: CSV_NAME_FIX.get(x, x))

    if 'higher_record' not in bracket.columns:
        wins_   = reg.groupby(['Season','WTeamID']).size().reset_index(name='Wins')
        losses_ = reg.groupby(['Season','LTeamID']).size().reset_index(name='Losses')
        rec_    = wins_.merge(losses_, left_on=['Season','WTeamID'], right_on=['Season','LTeamID'], how='outer')
        rec_['WinPct'] = rec_['Wins'].fillna(0) / (rec_['Wins'].fillna(0) + rec_['Losses'].fillna(0))
        rec_    = rec_.rename(columns={'WTeamID':'TeamID'})[['Season','TeamID','WinPct']]
        rec26_  = (rec_[rec_['Season']==2026]
                   .merge(teams[['TeamID','KenPomName']], on='TeamID')
                   .rename(columns={'KenPomName':'TeamName'})
                   .set_index('TeamName')['WinPct'])
        bracket['higher_record'] = bracket['higher_seed'].map(rec26_).fillna(0.5)
        bracket['lower_record']  = bracket['lower_seed'].map(rec26_).fillna(0.5)
        print('Win records added automatically from 2026 regular season data')

    # ── Build feature matrix ──
    stats_26 = box_stats[box_stats['Season'] == 2026].set_index('TeamName')
    bt_26    = bt_ratings[bt_ratings['Season'] == 2026].set_index('TeamName')
    missing  = []
    rows     = []

    for _, game in bracket.iterrows():
        h, l = game['higher_seed'], game['lower_seed']
        if h not in stats_26.index: missing.append(h)
        if l not in stats_26.index: missing.append(l)

        sh = stats_26.loc[h] if h in stats_26.index else pd.Series(dtype=float)
        sl = stats_26.loc[l] if l in stats_26.index else pd.Series(dtype=float)

        row = {
            'HigherSeedNum': game['higher_seed_num'],
            'LowerSeedNum':  game['lower_seed_num'],
            'HigherRecord':  game['higher_record'],
            'LowerRecord':   game['lower_record'],
        }
        stat_features = [f for f in features if f.endswith('_diff') and f != 'BT_diff']
        for col in stat_features:
            stat  = col.replace('_diff', '')
            h_val = sh.get(stat, np.nan)
            l_val = sl.get(stat, np.nan)
            row[col] = (h_val - l_val) if pd.notna(h_val) and pd.notna(l_val) else 0.0

        # BT_diff from bt_ratings
        if 'BT_diff' in features:
            bt_h = bt_26.loc[h, 'BT'] if h in bt_26.index else 0.0
            bt_l = bt_26.loc[l, 'BT'] if l in bt_26.index else 0.0
            row['BT_diff'] = bt_h - bt_l

        rows.append(row)

    if missing:
        print('WARNING: teams not found in 2026 box stats — predictions will use 0 for missing stats:')
        print(set(missing))
        print('Use find_team_name() in Cell Q to find the correct name')
        print()

    X     = pd.DataFrame(rows)[features].fillna(0)
    rf_p  = rf.predict_proba(X)[:, 1]
    xgb_p = xgb.predict_proba(X)[:, 1]

    predictions = (W_RF * rf_p + W_XGB * xgb_p).round(4)

    # Write predictions back into the original CSV's Predictions column
    original = pd.read_csv(matchup_csv_path)
    original['Predictions'] = predictions
    original.to_csv(matchup_csv_path, index=False)
    print(f'Updated Predictions column in {matchup_csv_path}')

    bracket['Prediction']       = predictions
    bracket['Predicted_Winner'] = np.where(
        predictions >= 0.5,
        bracket['higher_seed'],
        bracket['lower_seed']
    )
    bracket['Upset'] = predictions < 0.5
    return bracket

bracket = predict_bracket('2026_Potential_Matchups.csv')
pd.set_option('display.max_rows', None)
display(bracket[['higher_seed','higher_seed_num','lower_seed','lower_seed_num','Prediction']].head(10))

Loaded 2278 matchups
Win records added automatically from 2026 regular season data
Updated Predictions column in 2026_Potential_Matchups.csv


,higher_seed,higher_seed_num,lower_seed,lower_seed_num,Prediction
0,Alabama,4,Akron,12,0.6089
1,Arizona,1,Akron,12,0.8662
2,Arkansas,4,Akron,12,0.7891
3,BYU,6,Akron,12,0.7386
4,Akron,12,Cal Baptist,13,0.6644
5,Clemson,8,Akron,12,0.6997
6,Connecticut,2,Akron,12,0.8211
7,Duke,1,Akron,12,0.8704
8,Florida,1,Akron,12,0.7913
9,Akron,12,Furman,15,0.9486
